# Feature Exploration And Blind-Test Analysis

This notebook is set up for analysis that is useful in the report, not just quick model scoring.

Current evaluation rules:
- build window-level samples from each capture session
- cap windows per capture for balanced analysis
- train on sessions `00-04`
- test only on the untouched blind sessions `05`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from traffic_classifier.features import build_capture_summary, build_window_dataset
from traffic_classifier.modeling import (
    build_models,
    cap_windows_per_capture,
    evaluate_models,
    hold_out_session_id_per_label,
    random_forest_feature_importance,
    session_majority_vote,
    split_features_and_target,
    summarize_results,
)

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 100)

BLIND_TEST_SESSION_ID = 5
MAX_WINDOWS_PER_CAPTURE = 120
TRANSPORT_HEAVY_COLUMNS = [
    'udp_ratio', 'tcp_ratio', 'tls12_ratio', 'tls13_ratio', 'ipv6_ratio',
    'dst_443_ratio', 'src_443_ratio', 'outbound_ratio', 'inbound_ratio', 'direction_balance'
]

In [ ]:
capture_dir = Path('..') / 'Wireshark Captures'
feature_df = build_window_dataset(capture_dir, window_size=75, min_packets=40)
balanced_feature_df = cap_windows_per_capture(feature_df, MAX_WINDOWS_PER_CAPTURE)
capture_summary = build_capture_summary(capture_dir)

blind_split = hold_out_session_id_per_label(feature_df, BLIND_TEST_SESSION_ID)
balanced_blind_split = hold_out_session_id_per_label(balanced_feature_df, BLIND_TEST_SESSION_ID)

capture_summary.head()

## Capture Overview

In [ ]:
capture_summary[['capture_name', 'label', 'session_id', 'packet_count', 'duration_sec', 'top_protocols']].sort_values(['label', 'session_id'])

In [ ]:
blind_sessions = capture_summary[capture_summary['session_id'] == BLIND_TEST_SESSION_ID].copy()
blind_sessions[['capture_name', 'label', 'packet_count', 'duration_sec', 'top_protocols']].sort_values(['label', 'capture_name'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=capture_summary.sort_values(['label', 'session_id']), x='capture_name', y='packet_count', hue='label', dodge=False, ax=axes[0])
axes[0].set_title('Packets Per Capture Session')
axes[0].tick_params(axis='x', rotation=70)
axes[0].legend_.remove()

sns.barplot(data=capture_summary.sort_values(['label', 'session_id']), x='capture_name', y='duration_sec', hue='label', dodge=False, ax=axes[1])
axes[1].set_title('Duration Per Capture Session')
axes[1].tick_params(axis='x', rotation=70)
axes[1].legend(title='label', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()

In [ ]:
raw_window_counts = feature_df.groupby(['label', 'capture_name']).size().rename('raw_windows').reset_index()
balanced_window_counts = balanced_feature_df.groupby(['label', 'capture_name']).size().rename('balanced_windows').reset_index()
raw_window_counts.merge(balanced_window_counts, on=['label', 'capture_name'], how='left').sort_values(['label', 'capture_name'])

## Feature Relationships

In [ ]:
summary_cols = [
    'length_mean', 'length_std', 'length_p90', 'delta_mean', 'delta_std',
    'udp_ratio', 'tcp_ratio', 'inbound_ratio', 'outbound_ratio', 'direction_balance'
]
balanced_feature_df.groupby('label')[summary_cols].mean().round(3)

In [ ]:
corr_features = [
    'length_mean', 'length_std', 'length_p90', 'length_entropy',
    'delta_mean', 'delta_std', 'delta_p90', 'delta_fast_ratio',
    'udp_ratio', 'tcp_ratio', 'tls12_ratio', 'tls13_ratio',
    'inbound_ratio', 'outbound_ratio', 'direction_balance',
    'unique_destinations', 'packet_rate'
]
corr_matrix = balanced_blind_split.train_df[corr_features].corr().round(2)
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=True, fmt='.2f', square=True)
plt.title('Feature Correlation Heatmap On Training Sessions')
plt.tight_layout()

In [ ]:
upper_triangle_mask = pd.DataFrame(
    [[i >= j for j in range(len(corr_matrix.columns))] for i in range(len(corr_matrix.index))],
    index=corr_matrix.index,
    columns=corr_matrix.columns,
)
corr_pairs = corr_matrix.where(~upper_triangle_mask).stack().rename('correlation').reset_index()
corr_pairs.columns = ['feature_a', 'feature_b', 'correlation']
corr_pairs['abs_correlation'] = corr_pairs['correlation'].abs()
corr_pairs.sort_values('abs_correlation', ascending=False).head(15)

## Blind-Test Model Comparison

In [ ]:
train_X, y_train = split_features_and_target(balanced_blind_split.train_df)
test_X, y_test = split_features_and_target(balanced_blind_split.test_df)

models = build_models(random_state=42)
results = evaluate_models(models, train_X, test_X, y_train, y_test)
full_summary = summarize_results(results).assign(feature_set='balanced_full')

reduced_train_X = train_X.drop(columns=[col for col in TRANSPORT_HEAVY_COLUMNS if col in train_X.columns])
reduced_test_X = test_X.drop(columns=[col for col in TRANSPORT_HEAVY_COLUMNS if col in test_X.columns])

reduced_models = build_models(random_state=42)
reduced_results = evaluate_models(reduced_models, reduced_train_X, reduced_test_X, y_train, y_test)
reduced_summary = summarize_results(reduced_results).assign(feature_set='balanced_reduced')

pd.concat([full_summary, reduced_summary], ignore_index=True)[['feature_set', 'model', 'accuracy', 'macro_f1']]

In [ ]:
feature_ablation_sets = {
    'all_features': train_X.columns.tolist(),
    'no_capture_duration': [c for c in train_X.columns if c != 'capture_duration'],
    'no_duration_or_rate': [c for c in train_X.columns if c not in ['capture_duration', 'packet_rate']],
    'length_and_timing_only': [
        c for c in train_X.columns
        if c.startswith('length_') or c.startswith('delta_') or c in ['capture_duration', 'packet_rate']
    ],
}

ablation_rows = []
for feature_set_name, columns in feature_ablation_sets.items():
    ablation_models = build_models(random_state=42)
    ablation_results = evaluate_models(ablation_models, train_X[columns], test_X[columns], y_train, y_test)
    ablation_summary = summarize_results(ablation_results)
    rf_row = ablation_summary[ablation_summary['model'] == 'random_forest'].iloc[0]
    ablation_rows.append({
        'feature_set': feature_set_name,
        'n_features': len(columns),
        'accuracy': rf_row['accuracy'],
        'macro_f1': rf_row['macro_f1'],
    })

pd.DataFrame(ablation_rows).sort_values(['macro_f1', 'accuracy'], ascending=False).reset_index(drop=True)

In [ ]:
rf_result = [result for result in results if result.name == 'random_forest'][0]
plt.figure(figsize=(7, 6))
sns.heatmap(rf_result.confusion, annot=True, fmt='d', cmap='Blues')
plt.title('Blind-Test Confusion Matrix: Balanced Random Forest')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()

In [ ]:
rf_importances = random_forest_feature_importance(models['random_forest'], train_X.columns.tolist(), top_n=15)
rf_importances

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(data=rf_importances, x='importance', y='feature', palette='viridis')
plt.title('Top Random-Forest Features On Blind-Test Training Split')
plt.tight_layout()

In [ ]:
top_feature_names = rf_importances['feature'].head(8).tolist()
label_feature_means = balanced_blind_split.train_df.groupby('label')[top_feature_names].mean()
plt.figure(figsize=(10, 5))
sns.heatmap(label_feature_means, annot=True, fmt='.2f', cmap='mako')
plt.title('Per-Label Mean Values For Top Features')
plt.tight_layout()

## Session-Level Blind-Test Voting

In [ ]:
session_votes = session_majority_vote(models['random_forest'], balanced_blind_split.test_df)
session_votes

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=session_votes, x='capture_name', y='vote_share', hue='correct', dodge=False, palette={True: '#2a9d8f', False: '#e76f51'})
plt.axhline(0.5, color='black', linestyle='--', linewidth=1)
plt.ylim(0, 1.0)
plt.title('Session-Level Majority-Vote Confidence')
plt.ylabel('Vote Share For Predicted Label')
plt.xlabel('Capture Session')
plt.xticks(rotation=20)
plt.tight_layout()

## Notes For The Report

- Use the balanced blind-test random forest as the main result.
- Mention that the reduced-feature random forest stays close to the full-feature result.
- Mention that removing `capture_duration` does not hurt performance and can slightly improve it.
- Highlight that `walmart_05` is confused with `nbcnews`, which points to overlapping traffic behavior.
- Use the correlation table to justify dropping redundant features or discussing feature groups instead of listing dozens individually.
